# 01 — Scope 1 & 2 Baseline Calculator
**Purpose:** Calculate the GHG emissions baseline for the Lacq Gas Processing Site 
(illustrative) under Scope 1 (direct combustion) and Scope 2 (purchased electricity).

**Emission factors:** ADEME Base Empreinte® V23.10 (May 2026) — combustion-only, 
LHV basis, GWP100 AR5. Grid factor: RTE / ADEME FR_2023.

**Site data:** Illustrative, based on TotalEnergies Nouvelle-Aquitaine reporting 
ranges and IEA French petrochemical sector benchmarks (IEA, 2023).

**Author:** Emma McCallum  
**Date:** May 2026  
**Repository:** github.com/ecmccallum/industrial-decarbonization-scenarios

In [1]:
!pip install pandas numpy matplotlib seaborn scipy openpyxl duckdb --break-system-packages -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

In [3]:
from data.emission_factors import SCOPE1, SCOPE2_ELECTRICITY
from data.sites import LACQ, FRENCH_GAS_SECTOR_AVG

# Choose which site to analyse
site = LACQ

print(f"Analysing: {site['name']}")

Analysing: Lacq Gas Processing Site (illustrative)


In [4]:
def calculate_scope1(site, ef_scope1):
    """
    Calculate Scope 1 emissions from direct fuel combustion.
    Returns a dictionary of emissions by fuel type (tonnes CO2eq).
    """
    results = {}

    fuel_keys = {
        'natural_gas': 'natural_gas_MWh',
        'fuel_oil':    'fuel_oil_MWh',
        'coal':        'coal_MWh',
    }

    for fuel, site_key in fuel_keys.items():
        if site_key in site:
            mwh = site[site_key]
            ef = ef_scope1[fuel]        # kgCO₂eq / kWh
            tco2 = mwh * ef 
            results[fuel] = round(tco2, 1)

    if 'process_tCO2' in site:
        results['process_emissions'] = site['process_tCO2']

    results['TOTAL_scope1'] = round(sum(results.values()), 1)
    return results

scope1 = calculate_scope1(site, SCOPE1)
print('Scope 1 emissions (tCO2eq):')
for fuel, val in scope1.items():
    print(f'  {fuel:<25} {val:>10,.1f} tCO2eq')

Scope 1 emissions (tCO2eq):
  natural_gas                164,000.0 tCO2eq
  fuel_oil                    13,300.0 tCO2eq
  TOTAL_scope1               177,300.0 tCO2eq


In [5]:
def calculate_scope2(site, ef_electricity):
    """Calculate Scope 2 emissions — purchased electricity only."""
    grid = site['grid']
    ef = ef_electricity[grid]
    ef_ren = ef_electricity['renewable_ppa']
    mwh = site['electricity_MWh']

    return {
        'location_based': round(mwh * ef, 1),
        'market_based':   round(mwh * ef_ren, 1),
        'grid_ef_kgCO2_kWh': ef,
    }

scope2 = calculate_scope2(site, SCOPE2_ELECTRICITY)
print(f"Scope 2 location-based:  {scope2['location_based']:>10,.1f} tCO2eq")
print(f"Scope 2 market-based:    {scope2['market_based']:>10,.1f} tCO2eq")
print(f"Grid emission factor:    {scope2['grid_ef_kgCO2_kWh']:>10.3f} kgCO2eq/kWh")

Scope 2 location-based:    13,750.0 tCO2eq
Scope 2 market-based:       2,500.0 tCO2eq
Grid emission factor:         0.055 kgCO2eq/kWh


In [6]:
def make_summary_table(site, scope1, scope2):
    """Combine Scope 1 and 2 results into a pandas DataFrame."""
    rows = []

    for source, tco2 in scope1.items():
        if source != 'TOTAL_scope1':
            rows.append({
                'Scope':  'Scope 1',
                'Source': source.replace('_', ' ').title(),
                'tCO2eq': tco2,
                'Method': 'ADEME Base Empreinte V23.10',
            })

    rows.append({
        'Scope':  'Scope 2',
        'Source': 'Purchased electricity',
        'tCO2eq': scope2['location_based'],
        'Method': f"Location-based, {site['grid']} grid",
    })

    df = pd.DataFrame(rows)
    total = df['tCO2eq'].sum()
    df['% of total'] = (df['tCO2eq'] / total * 100).round(1)
    return df, total

df_summary, total_tco2 = make_summary_table(site, scope1, scope2)
print(df_summary.to_string(index=False))
print(f"\nTOTAL baseline: {total_tco2:,.0f} tCO2eq/year")

  Scope                Source   tCO2eq                       Method  % of total
Scope 1           Natural Gas 164000.0  ADEME Base Empreinte V23.10        85.8
Scope 1              Fuel Oil  13300.0  ADEME Base Empreinte V23.10         7.0
Scope 2 Purchased electricity  13750.0 Location-based, FR_2023 grid         7.2

TOTAL baseline: 191,050 tCO2eq/year
